# Introduction to Social Simulation

## What a regression can't tell you, and what an agent can

We start from a table you might actually be handed in practice: players, two demographic covariates (`age`, `sex`), and a final score from a Rock-Paper-Scissors tournament. We run a real regression on it - regression is a genuinely useful tool, and we're not pretending otherwise.

Then we ask: what produced these numbers? The regression can't answer that, because *how each player actually played* - their preferred move, their decision rule - was never in the table. Those are the modeler's assumptions about behavior, not something regression can recover from outcomes alone.

We then build two different, equally plausible stories about what `age` and `sex` might mean for how someone plays, run each one as agents, and see what a regression makes of each. The point isn't that agents replace regression. It's that a table of outcomes is compatible with more than one story about how it came to be, and regression cannot tell those stories apart.

# Part 0 - A table, and a real regression

This is what we're handed: one row per player, with `age`, `sex`, and `final_points` from a completed tournament. Nothing here says how anyone actually played.

In [1]:
import pandas as pd
from sklearn.linear_model import LinearRegression

observed = pd.read_csv('rps_players.csv')
observed

,player,age,sex,final_points
0,Ava,22,F,37
1,Ben,45,M,75
2,Cleo,29,F,31
3,Dan,51,M,31
4,Eva,34,F,27
5,Finn,19,M,56
6,Gia,60,F,48
7,Hugo,38,M,69


## Regressing final points on age and sex

We convert `sex` into a 0/1 column (`sex_M`, with `F` as the reference category) and fit a linear regression of `final_points` on `age` and `sex_M`. This is an ordinary, legitimate use of regression - we are not building a strawman to knock down.

In [2]:
X = pd.get_dummies(observed[['sex']], drop_first=True, dtype=int)
X['age'] = observed['age']
y = observed['final_points']

model = LinearRegression().fit(X, y)

pd.DataFrame({
    'term': ['Intercept'] + list(X.columns),
    'coefficient': [model.intercept_] + list(model.coef_)
})

,term,coefficient
0,Intercept,32.301924
1,sex_M,21.809761
2,age,0.095119


In [3]:
print(f'R-squared: {model.score(X, y):.3f}')

R-squared: 0.415


## What this regression tells us, and what it doesn't

The regression finds that `sex_M` matters a fair amount and `age` barely matters. If this were real data, a reasonable-sounding writeup might say: *"male players scored substantially higher, controlling for age."*

What it cannot say is *why*. Maybe men in this tournament happen to prefer a stronger move against what others played. Maybe age shapes how cautiously someone plays and sex is a coincidental proxy for something else entirely. The regression describes an association between the columns it was given - it has no way to tell us which of many possible behavioral stories, if any, actually produced it. To ask that question, we need to represent the behavior itself, not just its outcome.

# Part 1 - Story A: one way age and sex could shape behavior

Here is one assumption a modeler could make, connecting the covariates to the same three decision rules from Rock-Paper-Scissors:

- **`sex` decides the preferred move:** `F` prefers Rock, `M` prefers Paper.
- **`age` decides the decision rule:** under 30 plays `mostly_preferred`; 30 to 49 plays `always_preferred`; 50 and over plays `never_paper`.

This is an assumption, not a fact about real people - we made it up to have something concrete to simulate. The point of building it as agents is that, unlike the regression, we can now state exactly what each player is assumed to do, and watch the tournament that follows from it.

In [4]:
payoff = {
    ('Rock', 'Paper'): (0, 1),
    ('Paper', 'Rock'): (1, 0),
    ('Rock', 'Scissors'): (1, 0),
    ('Scissors', 'Rock'): (0, 1),
    ('Paper', 'Scissors'): (0, 1),
    ('Scissors', 'Paper'): (1, 0),
    ('Rock', 'Rock'): (0, 0),
    ('Paper', 'Paper'): (0, 0),
    ('Scissors', 'Scissors'): (0, 0)
}

def story_a(age, sex):
    """Turn a covariate pair into a (preferred_move, decision_rule) pair.
    This function IS the story: change it, and you change the mechanism."""
    preferred_move = 'Rock' if sex == 'F' else 'Paper'
    if age < 30:
        decision_rule = 'mostly_preferred'
    elif age < 50:
        decision_rule = 'always_preferred'
    else:
        decision_rule = 'never_paper'
    return preferred_move, decision_rule

## Building proto-agents from the story

Each player becomes a small dictionary. `preferred_move` and `decision_rule` are no longer given to us in a table - they are *assigned* by `story_a`, using only `age` and `sex`. This is the step a table alone can never show: turning a covariate into an assumption about behavior.

In [5]:
proto_society = []
for row in observed.itertuples(index=False):
    preferred_move, decision_rule = story_a(row.age, row.sex)
    proto_society.append({
        'name': row.player,
        'age': row.age,
        'sex': row.sex,
        'preferred_move': preferred_move,
        'decision_rule': decision_rule,
        'current_move': None,
        'score': 0
    })

pd.DataFrame(proto_society)[['name', 'age', 'sex', 'preferred_move', 'decision_rule']]

,name,age,sex,preferred_move,decision_rule
0,Ava,22,F,Rock,mostly_preferred
1,Ben,45,M,Paper,always_preferred
2,Cleo,29,F,Rock,mostly_preferred
3,Dan,51,M,Paper,never_paper
4,Eva,34,F,Rock,always_preferred
5,Finn,19,M,Paper,mostly_preferred
6,Gia,60,F,Rock,never_paper
7,Hugo,38,M,Paper,always_preferred


## Running the tournament

`choose_move` reads an agent's assigned rule and returns what it plays. `play_game` runs one match between two agents and updates both scores in place. We use `random.Random` as an explicit object (`rng`) so the whole run is tied to one reproducible seed.

In [6]:
from random import Random
from itertools import combinations

def choose_move(agent, rng):
    rule = agent['decision_rule']
    preferred = agent['preferred_move']

    if rule == 'always_preferred':
        move = preferred
    elif rule == 'never_paper':
        move = rng.choice(['Rock', 'Scissors'])
    elif rule == 'mostly_preferred':
        move = rng.choice([preferred, preferred, preferred, 'Rock', 'Paper', 'Scissors'])
    else:
        raise ValueError(f'Unknown decision rule: {rule}')

    agent['current_move'] = move
    return move

def play_game(player1, player2, payoff, rng):
    move1 = choose_move(player1, rng)
    move2 = choose_move(player2, rng)
    points1, points2 = payoff[move1, move2]
    player1['score'] += points1
    player2['score'] += points2

In [7]:
# 20 rounds, round-robin: every player meets every other player once per round
rng = Random(123)
for round_number in range(1, 21):
    for player1, player2 in combinations(proto_society, 2):
        play_game(player1, player2, payoff, rng)

story_a_results = pd.DataFrame(proto_society)[['name', 'age', 'sex', 'preferred_move', 'decision_rule', 'score']]
story_a_results

,name,age,sex,preferred_move,decision_rule,score
0,Ava,22,F,Rock,mostly_preferred,37
1,Ben,45,M,Paper,always_preferred,75
2,Cleo,29,F,Rock,mostly_preferred,31
3,Dan,51,M,Paper,never_paper,31
4,Eva,34,F,Rock,always_preferred,27
5,Finn,19,M,Paper,mostly_preferred,56
6,Gia,60,F,Rock,never_paper,48
7,Hugo,38,M,Paper,always_preferred,69


`rps_players.csv` was itself generated by running Story A once already (same assumption, same schedule, same seed). So the check below confirms the code faithfully reproduces the table we started from - it is not a new finding.

In [8]:
check = observed.merge(
    story_a_results[['name', 'score']], left_on='player', right_on='name'
)
check['matches_original_table'] = check['final_points'] == check['score']
check[['player', 'final_points', 'score', 'matches_original_table']]

,player,final_points,score,matches_original_table
0,Ava,37,37,True
1,Ben,75,75,True
2,Cleo,31,31,True
3,Dan,31,31,True
4,Eva,27,27,True
5,Finn,56,56,True
6,Gia,48,48,True
7,Hugo,69,69,True


# Part 2 - Story B: a different assumption, same covariates

Story A let `sex` decide the preferred move and `age` decide the decision rule. Story B swaps that around entirely:

- **`age` decides the preferred move:** under 40 prefers Rock, 40 and over prefers Paper.
- **`sex` decides the decision rule:** `F` plays `mostly_preferred`, `M` plays `never_paper`.

This is a genuinely different mechanism - not a small tweak. Nothing says either story is the "real" one; both are just assumptions a modeler could plausibly make, starting from the same two covariates.

In [9]:
def story_b(age, sex):
    preferred_move = 'Rock' if age < 40 else 'Paper'
    decision_rule = 'mostly_preferred' if sex == 'F' else 'never_paper'
    return preferred_move, decision_rule

story_b_society = []
for row in observed.itertuples(index=False):
    preferred_move, decision_rule = story_b(row.age, row.sex)
    story_b_society.append({
        'name': row.player, 'age': row.age, 'sex': row.sex,
        'preferred_move': preferred_move, 'decision_rule': decision_rule,
        'current_move': None, 'score': 0
    })

rng_b = Random(123)
for round_number in range(1, 21):
    for player1, player2 in combinations(story_b_society, 2):
        play_game(player1, player2, payoff, rng_b)

story_b_results = pd.DataFrame(story_b_society)[['name', 'age', 'sex', 'preferred_move', 'decision_rule', 'score']]
story_b_results

,name,age,sex,preferred_move,decision_rule,score
0,Ava,22,F,Rock,mostly_preferred,60
1,Ben,45,M,Paper,never_paper,36
2,Cleo,29,F,Rock,mostly_preferred,42
3,Dan,51,M,Paper,never_paper,34
4,Eva,34,F,Rock,mostly_preferred,51
5,Finn,19,M,Rock,never_paper,39
6,Gia,60,F,Paper,mostly_preferred,60
7,Hugo,38,M,Rock,never_paper,31


## Regressing Story B's outcomes on the same covariates

Same regression as Part 0 - `final_points` on `age` and `sex` - but now run on the scores that Story B's mechanism produced.

In [10]:
X_b = pd.get_dummies(story_b_results[['sex']], drop_first=True, dtype=int)
X_b['age'] = story_b_results['age']
y_b = story_b_results['score']

model_b = LinearRegression().fit(X_b, y_b)

pd.DataFrame({
    'term': ['Intercept'] + list(X_b.columns),
    'coefficient': [model_b.intercept_] + list(model_b.coef_)
})

,term,coefficient
0,Intercept,51.474305
1,sex_M,-18.347969
2,age,0.048985


In [11]:
print(f'R-squared: {model_b.score(X_b, y_b):.3f}')

R-squared: 0.725


## Two stories, two regressions - read side by side

| | Story A | Story B |
|---|---|---|
| What `sex` governs | preferred move | decision rule |
| What `age` governs | decision rule | preferred move |
| Regression: effect of being male | **positive** (about +22 points) | **negative** (about -18 points) |
| Regression: effect of age | small | small |

Both regressions are computed the same honest way, on real generated data, and both find that `sex` matters more than `age`. But they disagree about the *direction* of the sex effect - because "sex matters" was never really the finding. What mattered was which specific rule sex happened to be wired to in each story, and that wiring is an assumption we made, not something either regression discovered.

Someone shown only Story A's table would confidently write "being male raises your score." Someone shown only Story B's table would just as confidently write the opposite. Both would be accurately describing their regression. Neither would be describing a mechanism, because the mechanism was never in the table to begin with - it only existed in the rules we chose when we built the agents.

# Closing

A regression is not wrong here - both regressions above are correctly computed and both are informative about their own data. What they cannot do is adjudicate between Story A and Story B, or tell us whether either resembles whatever real process a real tournament's data came from. That requires either external evidence about how people actually play, or - as we did here - building the mechanism explicitly enough to run it and watch what it generates.

This is the sense in which agents let us represent *several* stories side by side, where a regression only ever describes the one table it was handed.